In [1]:
!pip -q install kiwipiepy nltk pandas pyarrow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 33.6 MB/s eta 0:00:00


In [2]:
import re
import math
import json
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Set

import pandas as pd
from tqdm import tqdm

import nltk
from nltk import word_tokenize, pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

from kiwipiepy import Kiwi
kiwi = Kiwi()
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

NLTK_PACKAGES = [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "wordnet",
    "omw-1.4",
    "stopwords"
]

for pkg in NLTK_PACKAGES:
    try:
        nltk.data.find(pkg)
    except LookupError:
        nltk.download(pkg)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
JSON_PATH = "복구된_논문_리스트.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

assert isinstance(raw, dict) and "NODE_LIST" in raw, "최상위에 'NODE_LIST' 키 XXX."
assert isinstance(raw["NODE_LIST"], list), "'NODE_LIST'는 list 형태여야 함"

df = pd.DataFrame(raw["NODE_LIST"])
assert "NODE_ID" in df.columns, "NODE_ID 컬럼XXX"

# 없을 수 있는 필드 보정
if "KYWD" not in df.columns:
    df["KYWD"] = ""
for col in ["ABST_KR", "ABST_EN", "NODE_TTLE", "NODE_TTLE_EN"]:
    if col not in df.columns:
        df[col] = ""

print("df shape:", df.shape)
print("sample columns:", df.columns.tolist()[:20])

df shape: (61990, 13)
sample columns: ['NODE_ID', 'IPRD_NM', 'PLCT_NM', 'NODE_TTLE', 'NODE_TTLE_EN', 'PBSH', 'NODE_LINK', 'NODE_CLSS_01', 'NODE_CLSS_02', 'AUTR_NM', 'KYWD', 'ABST_KR', 'ABST_EN']


In [4]:
def safe_str(x) -> str:
    if x is None:
        return ""
    try:
        if isinstance(x, float) and pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

# 괄호는 제거
# 용어(앞부분)는 그대로 두고
# 약어는 별도 토큰으로 분리해 붙일 수 있게 반환
PAREN_ABBR_RX = re.compile(
    r"(?P<head>[A-Za-z][A-Za-z0-9\s\-\+\/&]*?)\s*\(\s*(?P<abbr>[A-Za-z][A-Za-z0-9\-\+\/]{1,15})\s*\)"
)

def split_paren_abbr(text: str):
    """
    입력: text
    출력: (text_without_paren, extra_abbr_tokens)
      - text_without_paren: 'Gaussian Process Regression' 처럼 괄호 제거된 텍스트
      - extra_abbr_tokens: ['gpr'] 같이 약어 토큰 리스트
    """
    s = safe_str(text)
    if not s:
        return s, []

    abbrs = []
    def _repl(m):
        head = m.group("head").strip()
        abbr = m.group("abbr").strip().lower()
        if abbr:
            abbrs.append(abbr)
        return head  # 괄호 제거하고 head만 남김

    s2 = PAREN_ABBR_RX.sub(_repl, s)
    return s2, abbrs

In [5]:
RX_HAS_HANGUL = re.compile(r"[가-힣ㄱ-ㅎㅏ-ㅣ\u1100-\u11FF\u3130-\u318F]")
RX_HAS_LATIN  = re.compile(r"[A-Za-z]")

def normalize_kw_piece(s: str) -> str:
    """KYWD 공백/괄호 주변 정리"""
    s = safe_str(s).strip()
    if not s:
        return ""
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)
    return s

def normalize_kw_token_ko(s: str) -> str:
    """
    KO 키워드 토큰 1개 -> 진짜 토큰으로 만들기
    - 혼합(한글+영문)인 경우도 국문 토큰으로 처리(혼선 방지용..)
    - 공백은 한 개만 유지
    """
    s = normalize_kw_piece(s)
    if not s:
        return ""
    s = normalize_anchor_phrases(s)
    s = re.sub(r"\s+", " ", s)
    return s

def normalize_kw_token_en(s: str):
    s = safe_str(s).strip()
    if not s:
        return "", []

    # 괄호 약어 분리
    s2, abbrs = split_paren_abbr(s)

    # 공백/기호 정리
    s2 = re.sub(r"\s+", " ", s2).strip().lower()

    return s2, abbrs

def split_kywd_item_ko_en(item: str) -> Tuple[str, str]:
    """
    1개 KYWD 항목을 ko /en으로 분리.
    - 콤마는 이미 바깥에서 분리했으니 item 내부 공백은 유지(= 1개 구)
    - 괄호가 있으면 '괄호 밖/안'을 우선 분리
    - 그럼에도 한 항목에 한글+영문이 섞이면 -> ko에만 넣음(중복 최소화)
    """
    item = normalize_kw_piece(item)
    if not item:
        return "", ""

    # 1) 괄호 분리 우선
    m = re.match(r"^(.*?)\((.*?)\)$", item)
    if m:
        a = m.group(1).strip()
        b = m.group(2).strip()

        a_has_ko = bool(RX_HAS_HANGUL.search(a))
        a_has_en = bool(RX_HAS_LATIN.search(a))
        b_has_ko = bool(RX_HAS_HANGUL.search(b))
        b_has_en = bool(RX_HAS_LATIN.search(b))

        # 밖=한글, 안=영문
        if a_has_ko and not a_has_en and b_has_en and not b_has_ko:
            return a, b
        # 밖=영문, 안=한글
        if a_has_en and not a_has_ko and b_has_ko and not b_has_en:
            return b, a

        # 둘 다 같은 언어 성격이면: item 전체를 해당 언어로
        if a_has_ko or b_has_ko:
            return item, ""
        if a_has_en or b_has_en:
            return "", item

    # 2) 괄호 없거나 분리 실패 시: 혼합 여부 확인
    has_ko = bool(RX_HAS_HANGUL.search(item))
    has_en = bool(RX_HAS_LATIN.search(item))

    # 혼합이면 ko에 넣게
    if has_ko and has_en:
        return item, ""

    if has_ko:
        return item, ""
    if has_en:
        return "", item
    return "", item

def split_kywd_field(kywd_field: str) -> Tuple[List[str], List[str]]: # KYWD 전체를 콤마 기준으로 분리하고 각 항목 ko/en으로

    s = safe_str(kywd_field).strip()
    if not s:
        return [], []

    parts = [normalize_kw_piece(p) for p in s.split(",")]
    parts = [p for p in parts if p]

    ko_items, en_items = [], []
    for p in parts:
        ko, en = split_kywd_item_ko_en(p)

        if ko:
            ko_norm = normalize_kw_token_ko(ko) # KO
            if ko_norm:
                ko_items.append(ko_norm)

        if en:
            en_norm, abbrs = normalize_kw_token_en(en) # EN
            if en_norm:
                en_items.append(en_norm)

            en_items.extend(abbrs) # 괄호 약어 따로 토큰

    return ko_items, en_items

In [6]:
def normalize_text_pre_tokenize(s: str) -> str:
    s = safe_str(s)
    if not s:
        return ""

    # 앵커 먼저
    s = normalize_anchor_phrases(s)
    # 그 외 일반 구 결합(한국어 품질용)
    s = re.sub(r"딥\s*러닝", "딥러닝", s)
    s = re.sub(r"자연어\s*처리", "자연어처리", s)
    s = re.sub(r"강화\s*학습", "강화학습", s)

    return s

def normalize_anchor_phrases(s: str) -> str:
    """
    앵커 표현만 표준형으로 강제 수렴시키는 전용 정규화.
    - 일반 텍스트에는 영향 최소화
    - 앵커 변형(공백/하이픈/중점/언더스코어/대소문자/붙여쓰기/한글표기)을 흡수
    """
    if not s:
        return s

    # 중점/특수 점 -> 하이픈 후보로 통일
    s = re.sub(r"[·\u00B7]", "-", s)

    # 공백 정리
    s = re.sub(r"\s+", " ", s).strip()

    # === 앵커 규칙 테이블 ===
    # 대개 .. 공백을 '-' 치환
    RULES: List[Tuple[str, str, int]] = [
        # chat gpt / chat-gpt / chat_gpt + 버전 -> chat-gpt
        (r"\bchat\s*[-_ ]*\s*gpt(?:\s*[-_ ]?\s*\d+(?:\.\d+)?)?\b", "chat-gpt", re.IGNORECASE),
        (r"\bchatgpt(?:\s*[-_ ]?\s*\d+(?:\.\d+)?)?\b", "chat-gpt", re.IGNORECASE),
        # gpt / gpt-4 / gpt4 / gpt-4o / gpt-4.1 / gpt-3.5-turbo 등 -> gpt
        (r"\bgpt(?:\s*[-_ ]?\s*\d+(?:\.\d+)?(?:\s*[-_ ]?\s*(?:turbo|mini|nano|pro|plus|o|omni|vision))?)?\b", "gpt", re.IGNORECASE),

        # --- LLM / Large Language Model ---
        (r"\blarge\s*[-_ ]*\s*language\s*[-_ ]*\s*models?\b", "large-language-model", re.IGNORECASE),
        (r"\blanguage\s*[-_ ]*\s*generation\s*[-_ ]*\s*models?\b", "language-generation-model", re.IGNORECASE),
        # llm, sllm, llms, llm-based
        (r"\b(?:sllm|llms|llm)\b", "llm", re.IGNORECASE),
        (r"\bllm\s*[-_ ]*\s*based\b", "llm", re.IGNORECASE),

        # "챗지피티" 계열 한국어 (공백/하이픈/언더스코어 허용)
        (r"챗\s*[-_ ]*\s*지피티", "chat-gpt", 0),
        (r"챗지피티", "chat-gpt", 0),
        (r"챗-지피티", "chat-gpt", 0),

        # --- Multimodal model---
        (r"\bmultimodal\s*[-_ ]*\s*models?\b", "multimodal-model", re.IGNORECASE),
        (r"\bmulti\s*[-_ ]*\s*modal\s*[-_ ]*\s*models?\b", "multimodal-model", re.IGNORECASE),
        (r"멀티모달\s*모델", "multimodal-model", 0),
        (r"멀티\s*[-_ ]*\s*모달\s*모델", "multimodal-model", 0),

        # --- models 진짜 모델명들 ---
        (r"\bgemini(?:\s*[-_ ]?\s*\d+(?:\.\d+)?)?(?:\s*[-_ ]?\s*(?:pro|ultra|flash|nano))?\b", "gemini", re.IGNORECASE),
        (r"\bcopilot(?:\s*[-_ ]?\s*[a-z0-9]+)?\b", "copilot", re.IGNORECASE),
        (r"\bdeepseek(?:\s*[-_ ]?\s*(?:r\d+|v\d+|r1|v2|v3))?\b", "deepseek", re.IGNORECASE),
        (r"\bclaude(?:\s*[-_ ]?\s*\d+(?:\.\d+)?)?(?:\s*[-_ ]?\s*(?:opus|sonnet|haiku))?\b", "claude", re.IGNORECASE),
        (r"\bllama(?:\s*[-_ ]?\s*\d+(?:\.\d+)?)?(?:\s*[-_ ]?\s*\d+[a-z]+)?(?:\s*[-_ ]?\s*(?:instruct|chat))?\b", "llama", re.IGNORECASE),

        # HyperCLOVA X는 hyperclova-x
        (r"\bhyper\s*clova\s*[-_ ]*\s*x\b", "hyperclova-x", re.IGNORECASE),
        (r"\bhyperclova\s*[-_ ]*\s*x\b", "hyperclova-x", re.IGNORECASE),
        (r"\bhyperclova[-_ ]*x\b", "hyperclova-x", re.IGNORECASE),

        # dall-e 변형: dall e / dall-e / dall·e / dall_e
        (r"\bdall\s*[-_ ]*\s*e(?:\s*[-_ ]?\s*\d+(?:\.\d+)?)?\b", "dall-e", re.IGNORECASE),
        (r"\bdalle(?:\s*[-_ ]?\s*\d+(?:\.\d+)?)?\b", "dall-e", re.IGNORECASE),

        (r"\bmid\s*[-_ ]*\s*journey(?:\s*[-_ ]?\s*v?\d+(?:\.\d+)?)?\b", "midjourney", re.IGNORECASE),
        (r"\bmidjourney(?:\s*[-_ ]?\s*v?\d+(?:\.\d+)?)?\b", "midjourney", re.IGNORECASE),

        (r"\bstable\s*[-_ ]*\s*diffusion(?:\s*[-_ ]?\s*\d+(?:\.\d+)?)?\b", "stable-diffusion", re.IGNORECASE),

        # --- Generative AI / GenAI / 생성형AI ---
        (r"\bgenerative\s*[-_ ]*\s*ai\b", "generative-ai", re.IGNORECASE),
        (r"\bgen\s*[-_ ]*\s*ai\b", "genai", re.IGNORECASE),
        (r"\bgenai\b", "genai", re.IGNORECASE),

        (r"생성형\s*[-_ ]*\s*인공지능", "생성형-인공지능", 0),
        (r"생성형\s*[-_ ]*\s*ai", "생성형-ai", re.IGNORECASE),
    ]

    # 규칙 적용
    for pat, repl, flags in RULES:
        s = re.sub(pat, repl, s, flags=flags)

    return s

In [7]:
# 확인용 코드
tests = [
    "multi modal", "multi-modal", "multimodal", "multi·modal",
    "멀티 모달", "멀티-모달", "멀티모달",
    "chat gpt", "ChatGPT", "chat-gpt", "챗 지피티", "HyperCLOVA X",
    "gen ai", "genai", "Generative AI", "생성형 ai", "생성형인공지능",
]
for t in tests:
    print(t, "->", normalize_anchor_phrases(t))


multi modal -> multi modal
multi-modal -> multi-modal
multimodal -> multimodal
multi·modal -> multi-modal
멀티 모달 -> 멀티 모달
멀티-모달 -> 멀티-모달
멀티모달 -> 멀티모달
chat gpt -> chat-gpt
ChatGPT -> chat-gpt
chat-gpt -> chat-gpt
챗 지피티 -> chat-gpt
HyperCLOVA X -> hyperclova-x
gen ai -> genai
genai -> genai
Generative AI -> generative-ai
생성형 ai -> 생성형-ai
생성형인공지능 -> 생성형-인공지능


In [8]:
# KYWD가 없을 수도 있으니 보정
if "KYWD" not in df.columns:
    df["KYWD"] = ""

kw_parsed = df["KYWD"].apply(split_kywd_field)
df["KYWD_KO_LIST"] = kw_parsed.apply(lambda x: x[0])
df["KYWD_EN_LIST"] = kw_parsed.apply(lambda x: x[1])

df["KYWD_KR"] = df["KYWD_KO_LIST"].apply(lambda xs: " | ".join(xs) if xs else "")
df["KYWD_EN"] = df["KYWD_EN_LIST"].apply(lambda xs: " | ".join(xs) if xs else "")

RX_KO = re.compile(r"[가-힣]")

# 후보 구성
ko_cand = (
    df["NODE_TTLE"].map(safe_str) + " " +
    df["ABST_KR"].map(safe_str)
).str.strip()

en_cand = (
    df["NODE_TTLE_EN"].map(safe_str) + " " +
    df["ABST_EN"].map(safe_str)
).str.strip()

# 영문-only: ko_cand에 한글이 하나도 없는 경우
ko_has_ko = ko_cand.apply(lambda s: bool(RX_KO.search(s)))

# KO = 한글이 하나라도 있으면 유지(영문 섞여도 유지)
df["ko_text"] = ko_cand.where(ko_has_ko, "")

# EN = 기본은 en_cand
df["en_text"] = en_cand

# 영문-only 문서(ko_has_ko=False)에서만 NODE_TTLE(=ko_cand)을 EN에 보조로 포함
# (ABST_KR가 비어 있고 제목만 영문인 케이스 커버)
df.loc[~ko_has_ko, "en_text"] = (
    (df.loc[~ko_has_ko, "en_text"].map(safe_str).str.strip() + " " +
     df.loc[~ko_has_ko, "NODE_TTLE"].map(safe_str).str.strip())
    .str.strip()
)

print("[DEBUG] ko_text empty:", (df["ko_text"].str.len() == 0).sum())
print("[DEBUG] en_text empty:", (df["en_text"].str.len() == 0).sum())

[DEBUG] ko_text empty: 8012
[DEBUG] en_text empty: 10440


In [9]:
HANGUL_RX = re.compile(r"[가-힣]")
LATIN_RX  = re.compile(r"[A-Za-z]")

def hangul_ratio(s: str) -> float:
    s = safe_str(s)
    if not s:
        return 0.0
    h = len(HANGUL_RX.findall(s))
    l = len(LATIN_RX.findall(s))
    denom = h + l
    return (h / denom) if denom > 0 else 0.0

# 문서 판정용: 제목/초록/키워드까지 합친 문자열에서 한글 비율 계산
df["_lang_probe"] = (
    df["NODE_TTLE"].map(safe_str) + " " +
    df["NODE_TTLE_EN"].map(safe_str) + " " +
    df["ABST_KR"].map(safe_str) + " " +
    df["ABST_EN"].map(safe_str) + " " +
    df["KYWD"].map(safe_str)
).str.strip()

# 한글 비율이 매우 낮으면 "영어 위주 문서"로 간주 (5%)
TH_HANGUL = 0.05
df["is_english_doc"] = df["_lang_probe"].apply(hangul_ratio).lt(TH_HANGUL)

print("[DEBUG] is_english_doc True count:", df["is_english_doc"].sum())
print("[DEBUG] hangul_ratio describe:")
print(df["_lang_probe"].apply(hangul_ratio).describe())

[DEBUG] is_english_doc True count: 19384
[DEBUG] hangul_ratio describe:
count    61990.000000
mean         0.290837
std          0.317499
min          0.000000
25%          0.035312
50%          0.240320
75%          0.286950
max          1.000000
Name: _lang_probe, dtype: float64


In [10]:
STOP_KR = {
    # 불용어 처리
    "연구","방법","분석","결과","제안","적용","이용","기반","통해","사용",
    "실험","구현","평가","기술","문제","해결","기존","제시","가능",
    "활용","구성","요소","비교","최근","위해","대한","따라","또한","그리고","경우","방식",
    "논문","수행","기법","발생","이러","검증","고려","진행","대상"
}
STOP_EN = {
    "the","a","an","and","or","to","of","in","for","on","with","as","by",
    "we","our","this","that","is","are","was","were","be","been","being",
    "paper","study","method","result","proposed","based","using","use",
    "into","from","can","may","also"
}

EN_STOP = set(stopwords.words("english")) | STOP_EN

def is_stop_kr(w: str) -> bool:
    return w in STOP_KR

def is_stop_en(w: str) -> bool:
    return w in EN_STOP

def canonicalize_model_token(w: str) -> str:
    """
    토큰 단위에서의 앵커 최종 보정
    (normalize_anchor_phrases의 보조)
    """
    if w in {"chatgpt", "chat-gpt"}:
        return "chat-gpt"
    if w in {"multimodal", "multi-modal"}:
        return "multi-modal"
    return w

In [11]:
EN_IN_KO_TOKEN = re.compile(r"\b[A-Za-z][A-Za-z0-9]*(?:[-_][A-Za-z0-9]+)*\b")

SHORT_WHITELIST_KR = {"딥"} # 최소 예외(필요하면 더 추가)
ALLOW_KIWI_TAG_PREFIX = ("NN",) # 명사 계열
ALLOW_KIWI_TAG_EXACT  = {"SL", "XR"} # 외래어, 어근

def tokenize_ko(text: str) -> List[str]:
    """
    한국어 토큰화 (Kiwi)
    - 앵커/표현 정규화 후 형태소 분석
    """
    text = normalize_text_pre_tokenize(text)
    if not text:
        return []

    toks = list(kiwi.tokenize(text))
    out: List[str] = []

    i = 0
    while i < len(toks):
        t = toks[i]

        # 토큰 후보: 명사(NN*) / 외래어(SL) / 어근(XR) / 접두사(XPN)
        is_head_ok = (t.tag.startswith("NN") or t.tag in {"SL", "XR", "XPN"})
        if not is_head_ok:
            i += 1
            continue

        # --- 결합 대상: (명사+명사) or (접두사+명사) ---
        if i + 1 < len(toks):
            t2 = toks[i + 1]

            can_merge = (
                # 1) 명사 + 명사 (역 + 설계 -> 역설계)
                (t.tag.startswith("NN") and t2.tag.startswith("NN")) or

                # 2) 접두사 + 명사 (초 + 고속 -> 초고속)
                (t.tag == "XPN" and t2.tag.startswith("NN")) or

                # 3) 명사 + 명사형 접미사 (지향 + 성 -> 지향성)
                (t.tag.startswith("NN") and t2.tag == "XSN")
            )

            if can_merge:
                # 원문에서 공백이 없었던 경우에만 결합
                gap = text[t.end : t2.start]
                if gap == "":  # 완전히 붙어있을 때만 결합 허용
                    merged = (t.form + t2.form).strip().lower()

                    if len(merged) > 1 and not is_stop_kr(merged):
                        out.append(merged)
                        i += 2
                        continue

        # 결합이 안 되면: 단일 토큰 처리
        if t.tag == "XPN":
            i += 1
            continue

        if t.tag.startswith("NN") or t.tag in {"SL", "XR"}:
            w = t.form.strip().lower()
            if len(w) <= 1:
                i += 1
                continue
            if is_stop_kr(w):
                i += 1
                continue
            out.append(w)

        i += 1

    return out

In [12]:
def to_wordnet_pos(treebank_tag: str):
    if not treebank_tag:
        return None
    c = treebank_tag[0].upper()
    if c == "J":
        return wordnet.ADJ
    if c == "V":
        return wordnet.VERB
    if c == "N":
        return wordnet.NOUN
    if c == "R":
        return wordnet.ADV
    return None

S_ENDING_EXCEPT = {
    "analysis","class","physics","mathematics","statistics","series",
    "thesis","chaos","lens","glass","bias"
}
IRREG_PLURAL = {
    "indices":"index",
    "matrices":"matrix",
    "phenomena":"phenomenon",
    "criteria":"criterion",
}
ACRONYM_PLURAL_RX = re.compile(r"^[a-z]{2,8}s$")  # llms, cnns, gans ...

def postprocess_en_token(w: str) -> str:
    w = canonicalize_model_token(w.lower().strip())
    if not w:
        return w

    if w in IRREG_PLURAL:
        return IRREG_PLURAL[w]
    if w in S_ENDING_EXCEPT:
        return w

    # 약어 복수 처리: llms -> llm
    if ACRONYM_PLURAL_RX.match(w) and len(w) >= 3:
        return w[:-1]

    return w

EN_KEEP_RX = re.compile(r"^[a-z][a-z0-9\-]*$")

def tokenize_en(text: str, max_tokens: int = 3000) -> List[str]:
    """
    역할: en_text -> en_tokens 생성만 수행
    - 구 결합(PMI)은 후속 단계에서 처리
    """
    s = normalize_text_pre_tokenize(text)
    if not s:
        return []

    raw = word_tokenize(s)
    tagged = pos_tag(raw)

    toks: List[str] = []
    for w, tag in tagged:
        w = safe_str(w).lower()
        # 영문/숫자/하이픈만  남김
        w = re.sub(r"[^a-z0-9\-]", "", w)
        if not w:
            continue

        # 짧은 토큰 제거
        if len(w) <= 1:
            continue

        if is_stop_en(w):
            continue

        w = postprocess_en_token(w)
        if not w:
            continue

        wn_pos = to_wordnet_pos(tag)
        if wn_pos is not None:
            # 표제어화; 명사 복수/동사 변형 등 정리
            w = lemmatizer.lemmatize(w, pos=wn_pos)

        if not EN_KEEP_RX.match(w):
            continue

        toks.append(w)
        if len(toks) >= max_tokens:
            break

    return toks

In [13]:
_RX_NON_ALNUM = re.compile(r"[^0-9a-zA-Z가-힣]+")
_RX_SPACES = re.compile(r"\s+")

def norm_dedup_ko(tok:str) -> str:
    """
    whitespace는 1칸
    중점(·)은 공백
    끝의 구두점만 제거
    """
    t = safe_str(tok).strip()
    if not t:
        return ""
    t = re.sub(r"[·\u00B7]", " ", t) # · -> 공백
    t = re.sub(r"\s+", " ", t).strip() # 공백 정규화
    t = re.sub(r"[^\w\s]+$", "", t) # 끝 구두점 제거
    return t

def norm_dedup_en(tok: str) -> str:
    t = safe_str(tok).strip().lower()
    if not t:
        return ""
    t = re.sub(r"[·\u00B7]", " ", t) # · -> 공백
    t = re.sub(r"\s+", " ", t).strip() # 공백 정규화
    t = re.sub(r"[^\w\s]+$", "", t) # 끝 구두점 제거
    return t

_RX_NON_ALNUM = re.compile(r"[^0-9a-zA-Z가-힣]+")
_RX_SPACES = re.compile(r"\s+")

def norm_for_kw_match(s: str) -> str:
    s = safe_str(s).lower()
    s = _RX_NON_ALNUM.sub(" ", s)     # 괄호/구두점/하이픈 등 -> 공백
    s = _RX_SPACES.sub(" ", s).strip()
    return s

def kw_phrase_in_raw(kw_phrase: str, raw_text: str) -> bool:
    kw_n = norm_for_kw_match(kw_phrase)
    raw_n = norm_for_kw_match(raw_text)
    if not kw_n or not raw_n:
        return False
    pat = r"\b" + re.escape(kw_n) + r"\b"
    return re.search(pat, raw_n) is not None

def merge_kw_body_tokens(body_tokens, kw_tokens, is_ko: bool):
    """
    body_tokens: 제목/초록 토큰(이미 tokenize_ko/tokenize_en 결과)
    kw_tokens: split_kywd_field(KYWD)로 얻은 "phrase 그대로" 리스트
    is_ko: 한국어/영어 분기
    """
    out = []
    seen_norm = set()

    # 1) keyword phrase를 먼저 추가 (우선순위)
    for kw in (kw_tokens or []):
        kw_phrase = safe_str(kw).strip()
        if not kw_phrase:
            continue

        if is_ko:
            k_norm = norm_dedup_ko(kw_phrase)
            k_out  = kw_phrase # 공백 유지
        else:
            k_norm = norm_dedup_en(kw_phrase)
            k_out  = kw_phrase.lower() # 영문은 소문자

        if not k_norm:
            continue

        # 이미 키워드 내에서 중복이면 스킵
        if k_norm in seen_norm:
            continue

        out.append(k_out)
        seen_norm.add(k_norm)

    # 2) 본문 토큰 추가 (키워드와 중복이면 스킵됨)
    for t in (body_tokens or []):
        raw = safe_str(t).strip()
        if not raw:
            continue

        n = norm_dedup_ko(raw) if is_ko else norm_dedup_en(raw)
        if not n:
            continue

        # 이미 키워드/이전 토큰에서 본 것이면 스킵
        if n in seen_norm:
            continue

        out.append(raw if is_ko else raw.lower())
        seen_norm.add(n)

    return out

In [14]:
def has_hangul(s: str) -> bool:
    return bool(HANGUL_RX.search(safe_str(s)))

def has_latin(s: str) -> bool:
    return bool(LATIN_RX.search(safe_str(s)))

tqdm.pandas()

def build_row_tokens(row):
    # 원문 필드
    title_ko = safe_str(row.get("NODE_TTLE")).strip()
    abst_kr  = safe_str(row.get("ABST_KR")).strip()

    title_en = safe_str(row.get("NODE_TTLE_EN")).strip()
    abst_en  = safe_str(row.get("ABST_EN")).strip()

    # 키워드 split (phrase 리스트)
    ko_kw_tokens, en_kw_tokens = split_kywd_field(row.get("KYWD"))

    # 본문 텍스트 구성: 필드 단위로 국문/영문 여부 검사 후 포함
    ko_parts = []
    if has_hangul(title_ko):
        ko_parts.append(title_ko)
    if has_hangul(abst_kr):
        ko_parts.append(abst_kr)
    ko_main = " ".join(ko_parts).strip()

    en_parts = []
    if has_latin(title_en):
        en_parts.append(title_en)
    if has_latin(abst_en):
        en_parts.append(abst_en)
    en_main = " ".join(en_parts).strip()

    # 본문 토큰화 + merge (키워드 우선)
    ko_tokens = tokenize_ko(ko_main) if ko_main else []
    en_tokens = tokenize_en(en_main) if en_main else []

    ko_out = merge_kw_body_tokens(ko_tokens, ko_kw_tokens, is_ko=True)
    en_out = merge_kw_body_tokens(en_tokens, en_kw_tokens, is_ko=False)

    # 최종 토큰은 merge 결과 사용
    ko_tokens_final = ko_out
    en_tokens_final = en_out

    # 영문 국문 교집합/혼합 토큰
    ko_en_tokens = list(set(ko_tokens_final) & set(en_tokens_final))

    return ko_tokens_final, en_tokens_final, ko_en_tokens, ko_kw_tokens, en_kw_tokens

In [15]:
out = df.progress_apply(build_row_tokens, axis=1)

df["ko_tokens"] = out.apply(lambda x: x[0])
df["en_tokens"] = out.apply(lambda x: x[1])
df["ko_en_tokens"] = out.apply(lambda x: x[2])

# 디버깅용: KYWD가 실제로 원하는 형태로 들어갔는지 확인
df["ko_kw_tokens"] = out.apply(lambda x: x[3])
df["en_kw_tokens"] = out.apply(lambda x: x[4])

100%|██████████| 61990/61990 [17:22<00:00, 59.47it/s]


In [16]:
# 영문-only인데 ko_text가 비어있고 en_text는 채워지는지
mask_en_only = ~ko_has_ko
print(df.loc[mask_en_only, ["NODE_ID", "NODE_TTLE", "ABST_KR", "ABST_EN", "ko_text", "en_text"]].head(3))

# 한글+영문 혼합인데 ko_text가 유지되는지
mask_mixed = ko_has_ko & df["NODE_TTLE"].str.contains(r"[A-Za-z]", na=False)
print(df.loc[mask_mixed, ["NODE_ID", "NODE_TTLE", "ABST_KR", "ko_text"]].head(3))

         NODE_ID                                          NODE_TTLE ABST_KR  \
16  NODE10561419  Challenging Issues and Intelligent Protective ...           
18  NODE10561421  Operating Characteristics of the DC Circuit Br...           
19  NODE10561422  Analysis of blocking characteristics for each ...           

                                              ABST_EN ko_text  \
16  The MicroGrid (MG) is a power system that conn...           
18  Superconducting current limiting impedance sel...           
19  DC cutoff must be completed within a few ms. T...           

                                              en_text  
16  Challenging Issues and Intelligent Protective ...  
18  Operating Characteristics of the DC Circuit Br...  
19  Analysis of blocking characteristics for each ...  
        NODE_ID                                          NODE_TTLE ABST_KR  \
2  NODE10561232  Depth와 Amplitude Image를 이용한 차량 문 주변 장애물 인식 및 거...           
4  NODE10561238                    MEMS/FO

In [17]:
# 자주 같이 붙어서 나오는 단어 쌍들을 찾아서 하나의 토큰처럼 처리
# 즉 하나의 phase 토큰으로 쓰기 위해 찾는 것
def mine_bigrams_pmi(token_lists: List[List[str]], min_count=20, pmi_threshold=6.0, topk=200):
    uni = Counter()
    bi = Counter()
    total = 0

    for toks in token_lists:
        if not toks:
            continue
        total += len(toks)
        uni.update(toks)
        bi.update(zip(toks, toks[1:]))

    cand = []
    for (a, b), c_ab in bi.items():
        if c_ab < min_count:
            continue
        # 너무 짧거나 불용어면 제외
        if (len(a) <= 1) or (len(b) <= 1):
            continue
        if is_stop_kr(a) or is_stop_kr(b) or is_stop_en(a) or is_stop_en(b):
            continue

        # PMI 공식 log2( P(a,b) / (P(a) * P(b))) https://wikidocs.net/184237
        # 0에 가까울수록 a와 b는 독립적임 ..
        p_ab = c_ab / max(total, 1)
        p_a = uni[a] / max(total, 1)
        p_b = uni[b] / max(total, 1)
        pmi = math.log2(p_ab / max(p_a * p_b, 1e-12))

        if pmi >= pmi_threshold: # 임계치는 설정
            cand.append(((a, b), c_ab, pmi))

    cand.sort(key=lambda x: (-x[1], -x[2], x[0][0], x[0][1]))
    return cand[:topk]

def merge_bigrams(tokens: List[str], bigram_set: set, joiner: str): # 여기서 실제로 위에서 뽑은 후보 것들 하나 토큰으로 합침
    out = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) in bigram_set:
            out.append(tokens[i] + joiner + tokens[i+1])
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out

# ko 후보
ko_cand = mine_bigrams_pmi(df["ko_tokens"].tolist(), min_count=20, pmi_threshold=6.0, topk=200)
ko_bigram_set = set([x[0] for x in ko_cand])

print("\n[PHRASE DEBUG] ko PMI top 30")
for (a,b),c,pmi in ko_cand[:30]:
    print(f"  {a} + {b} | count={c} | pmi={pmi:.2f}")

# en 후보
en_cand = mine_bigrams_pmi(df["en_tokens"].tolist(), min_count=20, pmi_threshold=5.0, topk=200)
en_bigram_set = set([x[0] for x in en_cand])

print("\n[PHRASE DEBUG] en PMI top 30")
for (a,b),c,pmi in en_cand[:30]:
    print(f"  {a} + {b} | count={c} | pmi={pmi:.2f}")


[PHRASE DEBUG] ko PMI top 30
  기여 + 기대 | count=490 | pmi=7.23
  중요 + 역할 | count=327 | pmi=6.64
  neural + network | count=308 | pmi=10.92
  한계 + 극복 | count=297 | pmi=8.07
  설문조사 + 실시 | count=284 | pmi=8.25
  자료 + 수집 | count=271 | pmi=6.18
  기초 + 자료 | count=264 | pmi=7.64
  통계적 + 유의 | count=260 | pmi=8.69
  수집 + 자료 | count=255 | pmi=6.09
  전략 + 수립 | count=245 | pmi=7.87
  수요 + 증가 | count=220 | pmi=6.56
  자료 + spss | count=219 | pmi=7.55
  유의 + 차이 | count=218 | pmi=6.77
  디지털 + 트윈 | count=211 | pmi=9.74
  언어 + 모델 | count=203 | pmi=6.59
  spss + 프로그램 | count=201 | pmi=7.68
  도움 + 기대 | count=190 | pmi=6.86
  spss + win | count=178 | pmi=10.83
  생성형 + ai | count=175 | pmi=9.24
  특징 + 추출 | count=175 | pmi=6.69
  기초자료 + 제공 | count=171 | pmi=6.69
  비용 + 절감 | count=167 | pmi=8.49
  부족 + 실정 | count=160 | pmi=8.09
  convolutional + neural | count=156 | pmi=11.64
  pearson + correlation | count=153 | pmi=12.21
  중요성 + 강조 | count=150 | pmi=8.90
  투고 + 규정 | count=148 | pmi=10.85
  방안 + 마련 | count=1

In [18]:
df["ko_tokens"] = df["ko_tokens"].apply(lambda toks: merge_bigrams(toks, ko_bigram_set, joiner=" "))
df["en_tokens"] = df["en_tokens"].apply(lambda toks: merge_bigrams(toks, en_bigram_set, joiner=" "))

print("\n[DEBUG] After phrase-merge")
print("ko_tokens len:", df["ko_tokens"].apply(len).describe())
print("en_tokens len:", df["en_tokens"].apply(len).describe())


[DEBUG] After phrase-merge
ko_tokens len: count    61990.000000
mean        25.592918
std         25.612986
min          0.000000
25%          5.000000
50%         11.000000
75%         48.000000
max        213.000000
Name: ko_tokens, dtype: float64
en_tokens len: count    61990.000000
mean        52.450282
std         34.046627
min          0.000000
25%          9.000000
50%         61.000000
75%         76.000000
max        235.000000
Name: en_tokens, dtype: float64


In [19]:
def debug_samples(df: pd.DataFrame, n=5, seed=7):
    view = df.sample(min(n, len(df)), random_state=seed)

    print("\n[TOKEN DEBUG SAMPLE]")
    for _, r in view.iterrows():
        print("="*110)
        print("NODE_ID:", r["NODE_ID"])
        print("[NODE_TTLE]", safe_str(r.get("NODE_TTLE"))[:200])
        print("[NODE_TTLE_EN]", safe_str(r.get("NODE_TTLE_EN"))[:200])
        print("[ABST_KR head]", safe_str(r.get("ABST_KR"))[:200].replace("\n"," "))
        print("[ABST_EN head]", safe_str(r.get("ABST_EN"))[:200].replace("\n"," "))
        print("[KYWD]", safe_str(r.get("KYWD"))[:120])

        kt = (r.get("ko_tokens") or [])
        et = (r.get("en_tokens") or [])
        print("ko_tokens:", kt)
        print("en_tokens:", et)

def quality_checks(df: pd.DataFrame): # 자주 깨지는 것들만 추가로 .. 작성함 ...
    # 딥러닝 깨짐 러닝만 있고 딥/딥러닝이 없는 케이스
    bad_deep = df["ko_tokens"].apply(lambda t: ("러닝" in set(t or []) and "딥" not in set(t or []) and "딥러닝" not in set(t or []))).sum()

    # 멀티모달 깨짐 멀티+모달 있는데 멀티모달이 없는 케이스
    bad_multi = df["ko_tokens"].apply(lambda t: ("멀티" in set(t or []) and "모달" in set(t or []) and "멀티모달" not in set(t or []))).sum()

    # ko/en 교집합(혼재/중복 정도)
    def inter_size(r):
        return len(set(r["ko_tokens"]) & set(r["en_tokens"]))
    inter_stats = df.apply(inter_size, axis=1).describe()

    print("\n[TOKEN QUALITY CHECK]")
    print("cases: 러닝 only (no 딥/딥러닝):", int(bad_deep))
    print("cases: 멀티+모달 but no 멀티모달:", int(bad_multi))
    print("\n[ko ∩ en intersection size describe]")
    print(inter_stats)

debug_samples(df, n=30, seed=200)
quality_checks(df)


[TOKEN DEBUG SAMPLE]
NODE_ID: NODE11342775
[NODE_TTLE] Simple Dual-Feed Dual-Circular Polarization Antenna with High Isolation
[NODE_TTLE_EN] Simple Dual-Feed Dual-Circular Polarization Antenna with High Isolation
[ABST_KR head] 
[ABST_EN head] A dual-feed antenna system with circular polarization (CP) diversity and high isolation is proposed in this paper. The proposed antenna consists of two bent monopole antennas and an isosceles triangul
[KYWD] Circularly Polarized Antenna,Circular Polarization Diversity,Dual-Feed Antenna,High Isolation
ko_tokens: []
en_tokens: ['circularly polarized antenna', 'circular polarization diversity', 'dual-feed antenna', 'high isolation', 'simple', 'dual-feed', 'dual-circular', 'polarization', 'antenna', 'high', 'isolation', 'system', 'circular', 'cp', 'diversity', 'consist', 'two', 'bent', 'monopole', 'isoscele', 'triangular', 'partial', 'gnd', 'single', 'layer', 'simply', 'design', 'dualfeed', 'dual-cp', 'characteristic', 'bandwidth', 'bw', 'optimize'

In [20]:
def dedup_preserve_order(tokens: List[str]) -> List[str]:
    return list(dict.fromkeys(tokens))

df["merged_tokens"] = df.apply(lambda r: (r["ko_tokens"] or []) + (r["en_tokens"] or []) + (r["ko_en_tokens"] or []), axis=1)
df["merged_tokens_dedup"] = df["merged_tokens"].apply(dedup_preserve_order)

# 토큰 0개 논문 제거
before = len(df)
mask_nonempty = df["merged_tokens_dedup"].apply(lambda x: len(x) > 0)
df_empty = df.loc[~mask_nonempty].copy()   # 제거된 것 따로 보관
df = df.loc[mask_nonempty].copy()
after = len(df)

print(f"[FILTER] removed empty-token docs: {before - after} / {before} (remain={after})")

[FILTER] removed empty-token docs: 109 / 61990 (remain=61881)


In [21]:
# 토큰 개수 계산
df["token_len"] = df["merged_tokens_dedup"].apply(len)
df["token_len"].describe()

,token_len
count,61881.000000
mean,76.794557
std,49.411501
min,1.000000
25%,34.000000
50%,82.000000
75%,111.000000
max,355.000000


In [22]:
df_len1 = df[df["token_len"] == 1].copy()

# 길이 1짜리 리스트에서 유일 토큰 추출
df_len1["solo_token"] = df_len1["merged_tokens_dedup"].str[0]

TOPN = 50

solo_freq = (
    df_len1["solo_token"]
    .value_counts(dropna=False)
    .head(TOPN)
    .reset_index()
)

solo_freq.columns = ["solo_token", "doc_count"]

solo_freq

,solo_token,doc_count
0,학회소식,269
1,머리말,90
2,편집후기,87
3,행사일정,56
4,냉동공조 월호,52
5,학회일지,49
6,회보,49
7,학회 소식,47
8,논문지 투고규정,42
9,권두언,40


In [23]:
# token_len == 1 문서 중 "자주 등장하는 단일 토큰"만 제거
before = len(df)

MIN_FREQ = 3

# 제거 대상 단일 토큰 집합
drop_solo_tokens = set(
    solo_freq.loc[solo_freq["doc_count"] >= MIN_FREQ, "solo_token"]
)

mask_drop = (
    (df["token_len"] == 1) &
    (df["merged_tokens_dedup"].str[0].isin(drop_solo_tokens))
)

# 제거 대상 문서 저장 -> 확인용
df_len1_dropped = df.loc[mask_drop].copy()
df_len1_dropped[[
    "NODE_ID",
    "NODE_TTLE",
    "NODE_TTLE_EN",
    "KYWD",
    "merged_tokens_dedup",
    "token_len"
]].to_csv(
    "removed_token_len_eq_1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("[SAVED] removed_token_len_eq_1.csv", df_len1_dropped.shape)

# df 갱신
df = df.loc[~mask_drop].copy()

after = len(df)
print(f"[FILTER] removed frequent token_len==1 docs: {before - after} / {before} (remain={after})")

[SAVED] removed_token_len_eq_1.csv (1174, 29)
[FILTER] removed frequent token_len==1 docs: 1174 / 61881 (remain=60707)


In [24]:
OUT_PATH = "df_tokens.parquet"

keep_cols = [
    "NODE_ID", "PBSH",
    "NODE_CLSS_02",
    "NODE_TTLE", "NODE_TTLE_EN",
    "ABST_KR", "ABST_EN",
    "KYWD",
    "ko_text", "en_text",
    "ko_tokens", "en_tokens",
    "merged_tokens_dedup",
]

# 존재하는 컬럼만 저장
keep_cols = [c for c in keep_cols if c in df.columns]

df_out = df[keep_cols].copy()
df_out.to_parquet(OUT_PATH, index=False)

print("[SAVED]", OUT_PATH, "shape=", df_out.shape)

[SAVED] df_tokens.parquet shape= (60707, 13)


---

총 토큰화된 논문 수는 60707개 (61881)

(토큰 존재하는 61881개 논문들 중 토큰 1개 논문들 -> 그 중 3번 이상 나온 논문들 제거)

---



In [25]:
df = pd.read_parquet("df_tokens.parquet")
print(df.columns.tolist())

['NODE_ID', 'PBSH', 'NODE_CLSS_02', 'NODE_TTLE', 'NODE_TTLE_EN', 'ABST_KR', 'ABST_EN', 'KYWD', 'ko_text', 'en_text', 'ko_tokens', 'en_tokens', 'merged_tokens_dedup']
